# 📊 Business Intelligence Datasets – Getting Started

This notebook demonstrates how to load and perform basic exploratory analysis on each dataset in the [Business-iInteligence-dataset](https://github.com/Whiteknight54/Business-iInteligence-dataset) repository directly from GitHub — no download required.

## Datasets covered
| Dataset | Description |
|---------|-------------|
| **Sales** | 100 sales transactions with product, region, and revenue detail |
| **Customers** | 50 customer records with demographics, segments, and CLV |
| **Finance** | 11 quarters of P&L, cash flow, and balance-sheet metrics |
| **Marketing** | 29 campaign records with spend, impressions, conversions, and ROAS |
| **Supply Chain** | 50 procurement orders with inventory and delivery metrics |

## 0 · Setup
The cell below defines raw GitHub URLs for every dataset. Run it first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

BASE = 'https://raw.githubusercontent.com/Whiteknight54/Business-iInteligence-dataset/main/datasets'

URLS = {
    'sales':        f'{BASE}/sales/sales_data.csv',
    'customers':    f'{BASE}/customers/customer_data.csv',
    'finance':      f'{BASE}/finance/financial_data.csv',
    'marketing':    f'{BASE}/marketing/marketing_data.csv',
    'supply_chain': f'{BASE}/supply_chain/supply_chain_data.csv',
}
print('URLs configured ✓')

---
## 1 · Sales Data

In [ ]:
sales = pd.read_csv(URLS['sales'], parse_dates=['order_date'])
print(f'Shape: {sales.shape}')
sales.head()

In [ ]:
# Revenue by category
cat_rev = sales.groupby('category')['revenue'].sum().sort_values(ascending=False)
cat_rev.plot(kind='bar', title='Total Revenue by Product Category', color='steelblue')
plt.ylabel('Revenue (USD)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Monthly revenue trend
monthly = sales.set_index('order_date').resample('ME')['revenue'].sum()
monthly.plot(title='Monthly Revenue Trend', marker='o', color='darkorange')
plt.ylabel('Revenue (USD)')
plt.tight_layout()
plt.show()

print('\nTop 5 sales reps by total revenue:')
print(sales.groupby('sales_rep')['revenue'].sum().sort_values(ascending=False).head())

---
## 2 · Customer Data

In [ ]:
customers = pd.read_csv(URLS['customers'], parse_dates=['first_purchase_date', 'last_active_date'])
print(f'Shape: {customers.shape}')
customers.head()

In [ ]:
# Customer count by segment
seg_count = customers['segment'].value_counts()
seg_count.plot(kind='pie', autopct='%1.1f%%', title='Customers by Segment', ylabel='')
plt.tight_layout()
plt.show()

# Average CLV by loyalty tier
clv_tier = customers.groupby('loyalty_tier')['clv_score'].mean().sort_values(ascending=False)
clv_tier.plot(kind='bar', title='Average CLV Score by Loyalty Tier', color='mediumseagreen')
plt.ylabel('Avg CLV Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('\nTop 5 acquisition channels by customer count:')
print(customers['acquisition_channel'].value_counts().head())

---
## 3 · Financial Data

In [ ]:
finance = pd.read_csv(URLS['finance'])
print(f'Shape: {finance.shape}')
finance.head()

In [ ]:
# Revenue, gross profit, and net income over time
fig, ax = plt.subplots()
ax.plot(finance['period'], finance['total_revenue'], marker='o', label='Revenue')
ax.plot(finance['period'], finance['gross_profit'], marker='s', label='Gross Profit')
ax.plot(finance['period'], finance['net_income'], marker='^', label='Net Income')
ax.set_title('Quarterly P&L Overview')
ax.set_ylabel('USD')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Net margin trend
fig, ax2 = plt.subplots()
ax2.bar(finance['period'], finance['net_margin_pct'], color='mediumpurple')
ax2.set_title('Net Margin (%) by Quarter')
ax2.set_ylabel('Net Margin %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 4 · Marketing Data

In [ ]:
marketing = pd.read_csv(URLS['marketing'], parse_dates=['start_date', 'end_date'])
print(f'Shape: {marketing.shape}')
marketing.head()

In [ ]:
# Average ROAS by channel
channel_roas = marketing.groupby('channel')['roas'].mean().sort_values(ascending=False)
channel_roas.plot(kind='bar', title='Average ROAS by Marketing Channel', color='coral')
plt.ylabel('ROAS')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Budget vs revenue scatter
fig, ax = plt.subplots()
scatter = ax.scatter(marketing['spend_usd'], marketing['revenue_generated'],
                     c=marketing['roas'], cmap='RdYlGn', s=80, alpha=0.8)
plt.colorbar(scatter, label='ROAS')
ax.set_title('Campaign Spend vs Revenue Generated')
ax.set_xlabel('Spend (USD)')
ax.set_ylabel('Revenue Generated (USD)')
plt.tight_layout()
plt.show()

print('\nTop 5 campaigns by ROAS:')
print(marketing[['campaign_name', 'channel', 'spend_usd', 'revenue_generated', 'roas']]
      .sort_values('roas', ascending=False).head())

---
## 5 · Supply Chain Data

In [ ]:
supply = pd.read_csv(URLS['supply_chain'], parse_dates=['order_date'])
print(f'Shape: {supply.shape}')
supply.head()

In [ ]:
# On-time delivery rate by supplier
supplier_otd = (supply.groupby('supplier_name')['on_time_delivery']
                .apply(lambda x: (x == 'Yes').mean() * 100)
                .sort_values(ascending=False))
supplier_otd.plot(kind='bar', title='On-Time Delivery Rate by Supplier (%)', color='teal')
plt.ylabel('On-Time Delivery %')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Average lead time by category
lead_time = supply.groupby('category')['lead_time_days'].mean().sort_values(ascending=False)
lead_time.plot(kind='barh', title='Average Lead Time by Product Category (days)', color='sandybrown')
plt.xlabel('Lead Time (days)')
plt.tight_layout()
plt.show()

print('\nTotal spend by supplier:')
print(supply.groupby('supplier_name')['total_cost'].sum().sort_values(ascending=False))

---
## 6 · Next Steps

Now that you have the data loaded, here are some ideas for deeper analysis:

- **Sales × Customers join** – enrich transactions with customer segment and CLV
- **Marketing attribution** – map campaign spend to downstream revenue in the sales dataset
- **Supply chain risk** – flag suppliers with low on-time delivery or quality pass rates
- **Financial forecasting** – fit a linear or ARIMA model to the quarterly P&L data
- **Dashboards** – export to Power BI, Tableau, or Looker Studio using the raw CSV URLs

All datasets are available as raw CSVs at:
```
https://raw.githubusercontent.com/Whiteknight54/Business-iInteligence-dataset/main/datasets/<folder>/<file>.csv
```